# just_drop vs just_drop_hal：处理 PPS

**比什么。** 同一块 ESP32-C3、同一 AP、同一 TR3000 发包机上，比较两条数据面：HAL（`just_drop_hal`，应用层 UDP:8080 主动丢弃并计 `drop`）与 XDP（`just_drop` 固件 + 冷表 `just_drop.elf`，Wi-Fi 入口 XDP_DROP 计 `drop`）。指标是进入各自丢弃路径的处理 PPS，不是无线物理层接收 PPS；驱动在计数点之前丢掉的帧看不见。

**怎么跑。** Mac 只开 `cargo xtask agent --listen 0.0.0.0:9240 --no-monitor`（cwd=`xpost/`）；收到烧录请求时按 `profiles.toml` 的 `build` 编译再烧。TR3000 上 `./xpost-test`（编译期嵌入 `config/test.toml`：默认 3 对、每臂 40 s、1 s 采样、`formal=true`）。每一对先烧 HAL 再烧 XDP：agent `FlashProfile` → 等启动 → `ping` → HAL 再发 `drop on` → 开测。库写入 schema `formal` 的 `experiment_*`，开跑前清空表。本笔记本只读库做性能汇总与图，不烧录、不发包。

**包怎么发。** TR3000 用 AF_PACKET RAW 向 DUT 的 `esp_ip:8080` 打满速 UDP（固定小包、多线程 `udp_blast`），源 IP 伪造为同网段地址；窗口默认 40 s，主机侧记录成功提交发送数 → `host_pps`。DUT 不解析 payload；串口只读单调 `drop`（及对照用的 `net_rx`）。XDP 臂要求窗口内 `net_rx` 不增（包未进协议栈）；任一臂 `drop` 不增则失败。

**数怎么记、怎么算。** 发包开始前读一次 `counters` 作起点，窗口内约每 1 s 再读，结束后再读终点；原始行是 `(run_id, time_ns, counter_value)`，`time_ns` 相对该轮主机单调钟。单轮平均处理 PPS = `(C_end−C_start)/(t_end−t_start)`（计数下降整轮作废，不用饱和减法抹成 0）。跨轮报均值、样本标准差与 `CV=SD/mean`。当前是等间隔短窗口，记 `run_kind=pilot`：可看量级与臂间差，不能当作规范里的随机三点 Allan / 噪声模型正式结论（那些要等 triad 采集把 `sample_reads` 填上）。图 1：单绘图区、X 仅 HAL|XDP，左 Mean PPS、中 1 s 相对 ADEV（无三点则 n/a）、右跨轮 CV%；图 2：噪声诊断，数据不足则标证据不足。


In [ ]:
from collections import defaultdict
from utils.db import dsn_summary
from utils.experiment import (
    load_runs, load_counter_samples, load_sample_reads,
    mean_pps, summarize_variant, figure1_summary, figure2_noise,
)

print(dsn_summary())
runs = load_runs(run_kind="pilot")
samples = load_counter_samples(runs["run_id"].tolist() if len(runs) else [])
reads = load_sample_reads(runs["run_id"].tolist() if len(runs) else [])
display(runs)
print(f"samples={len(samples)} triad_reads={len(reads)}")

by_variant = defaultdict(list)
for _, row in runs.iterrows():
    sub = samples[samples["run_id"] == row["run_id"]].sort_values("time_ns")
    by_variant[row["variant"]].append(
        mean_pps(
            sub["time_ns"].tolist(), sub["counter_value"].tolist(),
            run_id=row["run_id"], variant=row["variant"], pair_id=int(row["pair_id"]),
        )
    )
summaries = {v: summarize_variant(rs) for v, rs in by_variant.items()}
for v, s in summaries.items():
    print(f"{v}: valid={s['n_valid']}/{s['n_total']}  "
          f"mean={s['mean_pps']:.0f}  sd={s['sd_pps']:.0f}  CV={s['cv']:.4f}  "
          f"per-run={s['per_run_pps']}")

hal, xdp = summaries.get("hal", {}), summaries.get("xdp", {})
fig1, _ = figure1_summary(
    hal_mean_pps=hal.get("mean_pps", float("nan")),
    xdp_mean_pps=xdp.get("mean_pps", float("nan")),
    hal_pps_points=hal.get("per_run_pps"),
    xdp_pps_points=xdp.get("per_run_pps"),
    hal_adev_1s=None, xdp_adev_1s=None,
    hal_cv=hal.get("cv"), xdp_cv=xdp.get("cv"),
    title="Pilot PPS / ADEV(1s n/a) / CV",
)
display(fig1)
display(figure2_noise(
    title="Noise diagnostics — awaiting random triad runs",
))
